<a href="https://colab.research.google.com/github/Svein-Tore/colab/blob/main/FOPDT-binder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [89]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import FloatSlider, Button, VBox, HBox, FileUpload, Output
from IPython.display import display, Markdown, HTML, clear_output
import io
import base64

# === 1. Oppsett ===
uploader = FileUpload(accept='', multiple=False, description="Last opp måledata")
app_display = Output()

def start_analysen(change):
    with app_display:
        clear_output(wait=True)
        if not uploader.value: return

        # Robust filhenting (Fikset for ipywidgets 8)
        raw = uploader.value
        try:
            file_item = raw[0] if isinstance(raw, (list, tuple)) else list(raw.values())[0]
            content = file_item['content']
            df = pd.read_csv(io.BytesIO(content), sep=None, engine='python', decimal=',')
            t_d, n_d = df.iloc[:,0].values, df.iloc[:,1].values
        except:
            print("Feil: Sjekk CSV-formatet.")
            return

        # === 2. ESTIMERING (10/85-regel) ===
        y0_v = n_d[0]
        A_v = n_d[-1] - y0_v
        t10 = t_d[np.where(n_d > y0_v + 0.10 * A_v)[0][0]]
        t85 = t_d[np.where(n_d > y0_v + 0.85 * A_v)[0][0]]
        t63 = t_d[np.where(n_d > y0_v + 0.63 * A_v)[0][0]]
        L_est = max(0, float(t10 - 0.05 * (t85 - t10)))
        T_est = max(0.1, float(t63 - L_est))

        # === 3. PLOTT-FUNKSJON MED ALLE LINJER ===
        plot_out = Output()

        def update_plot(change=None):
            with plot_out:
                clear_output(wait=True)
                A, T, L, y0 = A_s.value, T_s.value, L_s.value, y0_s.value
                y_m = np.where(t_d < L, y0, y0 + A * (1 - np.exp(-(t_d - L) / T)))

                fig, ax = plt.subplots(figsize=(8, 5))
                ax.plot(t_d, n_d, "b.", markersize=3, alpha=0.3, label="Måledata")
                ax.plot(t_d, y_m, "r-", linewidth=2, label="FOPDT Modell")

                # Pedagogiske hjelpelinjer
                ax.axhline(y0, color='black', linestyle='--', alpha=0.4)
                ax.text(t_d[0], y0, f' y0={y0:.1f}', fontweight='bold', va='bottom')
                ax.axvline(L, color='orange', linestyle=':', linewidth=2)
                ax.text(L+110, y0+30, f' L={L:.1f}s', color='orange', fontweight='bold', ha='right')

                y63 = y0 + 0.63*A
                ax.plot(L+T, y63, 'go')
                ax.text(L+T, y63, f' T={T:.1f}s', color='green', fontweight='bold', va='top')

                ax.vlines(t_d[-1], y0, y0+A, color='purple', linewidth=3)
                ax.text(t_d[-1], y0+A/2, f' Δy={A:.1f}', color='purple', fontweight='bold')

                ax.grid(True, which='both', linestyle='--', alpha=0.5)
                ax.set_xlabel("Tid [s]"); ax.set_ylabel("Nivå")
                ax.legend(loc='lower right')
                plt.show()

        # === 4. WIDGETS ===
        style = {'description_width': 'initial'}
        A_s = FloatSlider(value=A_v, min=A_v*0.2, max=A_v*2, step=0.01, description="Δy", style=style)
        T_s = FloatSlider(value=T_est, min=0.1, max=T_est*4, step=0.1, description="T", style=style)
        L_s = FloatSlider(value=L_est, min=0, max=t_d[-1]/2, step=0.1, description="L", style=style)
        y0_s = FloatSlider(value=y0_v, min=y0_v-10, max=y0_v+10, step=0.01, description="y0", style=style)

        for s in [A_s, T_s, L_s, y0_s]: s.observe(update_plot, "value")

        # === 5. SKUDDSIKKER TABELL OG FORMEL (HTML) ===
        # Her tvinger vi frem sentrering og riktig utseende
        info_html = r"""
        <div style="font-family: sans-serif; padding: 20px; border: 1px solid #ddd; border-radius: 10px; background: #fafafa; min-width: 400px; margin-left: 20px;">
            <h3 style="text-align: center; color: #333;">SIMC Reguleringstabell</h3>
            <table style="width: 100%; border-collapse: collapse; background: white;">
                <thead>
                    <tr style="background: #eee;">
                        <th style="padding: 10px; border: 1px solid #ccc;">Valg av λ</th>
                        <th style="padding: 10px; border: 1px solid #ccc;">Respons</th>
                        <th style="padding: 10px; border: 1px solid #ccc;">Observasjon</th>
                    </tr>
                </thead>
                <tbody>
                    <tr><td style="padding: 8px; border: 1px solid #ccc; text-align: center;">T / 2</td><td style="padding: 8px; border: 1px solid #ccc;">Rolig</td><td style="padding: 8px; border: 1px solid #ccc;">Lite oversving</td></tr>
                    <tr><td style="padding: 8px; border: 1px solid #ccc; text-align: center;">T / 4</td><td style="padding: 8px; border: 1px solid #ccc;">Standard</td><td style="padding: 8px; border: 1px solid #ccc;">God balanse</td></tr>
                    <tr><td style="padding: 8px; border: 1px solid #ccc; text-align: center;">T / 6</td><td style="padding: 8px; border: 1px solid #ccc;">Rask</td><td style="padding: 8px; border: 1px solid #ccc;">Aggressiv</td></tr>
                </tbody>
            </table>
            <br>
            <div style="background: #fff; padding: 15px; border-radius: 5px; border: 1px solid #eee;">
                <h4 style="margin-top: 0;">PID Formler</h4>
                <p style="font-size: 1.1em; margin: 5px 0;"><b>K = Δy / Δu</b></p>
                <p style="font-size: 1.1em; margin: 5px 0;"><b>Kp = T / (K · (λ + L))</b></p>
                <p style="font-size: 1.1em; margin: 5px 0;"><b>Ti = min(T, 4 · (λ + L))</b></p>
            </div>
        </div>
        """
        info_widget = widgets.HTML(value=info_html)

        # Layout
        kontroller = VBox([A_s, T_s, L_s, y0_s], layout={'width': '250px', 'padding': '20px 0'})
        dashbord = HBox([plot_out, kontroller, info_widget], layout={'flex_flow': 'row wrap', 'align_items': 'flex-start'})

        display(Markdown(f"**Auto-analyse ferdig.** Juster sliderne for å fininnstille."), dashbord)
        update_plot()

# === Start ===
uploader.observe(start_analysen, names='value')
display(Markdown("# FOPDT Simulator - Identifikasjon"), uploader, app_display)


# FOPDT Simulator - Identifikasjon

FileUpload(value={}, description='Last opp måledata')

Output()